In [1]:
import serial
import pandas as pd
import altair as alt
import time
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, FileLink

# setup Altair renderer
alt.renderers.enable('default')
alt.data_transformers.disable_max_rows()

# serial port
ser = serial.Serial('COM5', 115200, timeout=1)
time.sleep(2)

# control
chart_output = widgets.Output()
text_output = widgets.Output()
display(chart_output)
display(text_output)

# initial
data = []
id_count = defaultdict(int)
display_interval = 0.5
last_update = time.time()

try:
    while True:
        line = ser.readline().decode('utf-8').strip()
        if ',' in line:
            try:
                parts = line.split(',')
                face_id = int(parts[0])
                x = int(parts[1])
                y = int(parts[2])

                id_count[face_id] += 1
                data.append({"id": str(face_id), "x": x, "y": y})

                now = time.time()
                if now - last_update > display_interval:
                    df = pd.DataFrame(data)

                    chart = alt.Chart(df).mark_circle(size=80).encode(
                        x=alt.X('x:Q', scale=alt.Scale(domain=[0, 320])),
                        y=alt.Y('y:Q', scale=alt.Scale(domain=[240, 0])),
                        color='id:N',
                        tooltip=['id', 'x', 'y']
                    ).properties(
                        width=400,
                        height=300,
                        title='HuskyLens Face Tracking'
                    ).interactive()

                    with chart_output:
                        chart_output.clear_output(wait=True)
                        display(chart)

                    with text_output:
                        text_output.clear_output(wait=True)
                        print("Statistics of occurrence frequency:")
                        for fid, count in id_count.items():
                            print(f"ID {fid}: {count} times")

                    last_update = now

            except Exception as e:
                with text_output:
                    print("Parsing error", e)

except KeyboardInterrupt:
    print("Halt grab")
    ser.close()

    # save CSV
    if data:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        filename = f"face_data_{timestamp}.csv"
        pd.DataFrame(data).to_csv(filename, index=False)
        print(f"The data has been saved as CSV file：{filename}")
        display(FileLink(filename, result_html_prefix="Click to download: "))
    else:
        print("No data, no generated file")


Output()

Output()

Halt grab
The data has been saved as CSV file：face_data_20250429_202151.csv


C:\Users\86131\MFG598\finalProject\face_data_20250429_202151.csv